In [1]:
import math
import numpy as np
import scipy.stats as st
import scipy.optimize as opt

np.random.seed(52)

In [2]:
alpha = 0.05
m = 10
n = 100

digits = np.arange(m)
obs = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])

In [3]:
def kolmogorov_cdf(x, terms=2000):
    s = 1.0
    for j in range(1, terms + 1):
        s += 2 * (-1) ** j * np.exp(-2 * j * j * x * x)
    return s


def normal_grouped_nll(params):
    mu, sigma = params
    if sigma <= 0:
        return np.inf

    cuts = st.norm.cdf(digits[1:], loc=mu, scale=sigma)
    probs = np.empty(m)
    probs[0] = cuts[0]
    probs[1:-1] = np.diff(cuts)
    probs[-1] = 1.0 - cuts[-1]

    if np.any(probs <= 0):
        return np.inf

    return -np.sum(obs * np.log(probs))


def norm_cdf(x, mu, sigma):
    return 0.5 * (1.0 + math.erf((x - mu) / (np.sqrt(2) * sigma)))


def lilliefors_bootstrap(mu, sigma, rep=5000):
    ans = np.empty(rep)
    model = st.norm(loc=mu, scale=sigma)

    for b in range(rep):
        y = np.sort(model.rvs(size=n))
        mu_b = y.mean()
        sigma_b = y.std(ddof=1)
        ecdf = np.arange(n + 1) / n

        stat = 0.0
        for t in range(n):
            f = norm_cdf(y[t], mu_b, sigma_b)
            stat = max(stat, np.sqrt(n) * max(abs(f - ecdf[t]), abs(f - ecdf[t + 1])))

        ans[b] = stat

    return ans

In [4]:
exp_uniform = np.full(m, n / m)
chi2_uniform = float(((obs - exp_uniform) ** 2 / exp_uniform).sum())
p_uniform_chi2 = float(1 - st.chi2(df=m - 1).cdf(chi2_uniform))

print('Равномерная модель U{0,1,...,9}, критерий χ²')
print(f'chi^2 = {chi2_uniform:.6f}; p-value = {p_uniform_chi2:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_uniform_chi2 > alpha else 'отвергается')

emp_cdf = np.array([obs[:i].sum() for i in range(m + 1)]) / n
theor_uniform_cdf = np.arange(m + 1) / m

D_uniform = float(
    np.sqrt(n) * max(
        max(abs(emp_cdf[i] - theor_uniform_cdf[i]), abs(emp_cdf[i + 1] - theor_uniform_cdf[i]))
        for i in range(m)
    )
)
p_uniform_ks = float(1 - kolmogorov_cdf(D_uniform))

print()
print('Равномерная модель U{0,1,...,9}, критерий Колмогорова')
print(f'Δ = {D_uniform:.6f}; p-value = {p_uniform_ks:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_uniform_ks > alpha else 'отвергается')

fit = opt.differential_evolution(
    func=normal_grouped_nll,
    bounds=[(0, 10), (1e-6, 10)],
    maxiter=10000,
    seed=52
)

mu_hat = float(fit.x[0])
sigma_hat = float(fit.x[1])

cuts = st.norm.cdf(digits[1:], loc=mu_hat, scale=sigma_hat)
exp_normal = np.empty(m)
exp_normal[0] = n * cuts[0]
exp_normal[1:-1] = n * np.diff(cuts)
exp_normal[-1] = n * (1 - cuts[-1])

chi2_normal = float(((obs - exp_normal) ** 2 / exp_normal).sum())
p_normal_chi2 = float(1 - st.chi2(df=m - 1 - 2).cdf(chi2_normal))

print()
print('Нормальная модель N(μ, σ²), критерий χ²')
print(f'μ = {mu_hat:.6f}; σ = {sigma_hat:.6f}')
print(f'chi^2 = {chi2_normal:.6f}; p-value = {p_normal_chi2:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_normal_chi2 > alpha else 'отвергается')

sample = np.repeat(digits, obs)
mu0 = float(sample.mean())
sigma0 = float(sample.std(ddof=1))

D_normal = float(
    max(
        np.sqrt(n) * max(
            abs(norm_cdf(digits[i], mu0, sigma0) - emp_cdf[i]),
            abs(norm_cdf(digits[i], mu0, sigma0) - emp_cdf[i + 1])
        )
        for i in range(m)
    )
)

print()
print('Нормальная модель N(μ, σ²), критерий Колмогорова')
print(f'Δ = {D_normal:.6f}')

boot_stats = lilliefors_bootstrap(mu0, sigma0, rep=5000)
p_normal_ks = float(np.mean(boot_stats >= D_normal))

print(f'p-value = {p_normal_ks:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_normal_ks > alpha else 'отвергается')

print()
print('Сравнение результатов:')
print('1) Для равномерного распределения критерий χ² гипотезу не отвергает, а критерий Колмогорова — отвергает.')
print('2) Для нормального распределения критерий χ² также не отвергает гипотезу, а критерий Колмогорова — отвергает.')

Равномерная модель U{0,1,...,9}, критерий χ²
chi^2 = 16.400000; p-value = 0.0590
Решение по H0: нет оснований для отклонения гипотезы

Равномерная модель U{0,1,...,9}, критерий Колмогорова
Δ = 1.400000; p-value = 0.0397
Решение по H0: отвергается

Нормальная модель N(μ, σ²), критерий χ²
μ = 5.289676; σ = 2.679518
chi^2 = 9.802558; p-value = 0.2000
Решение по H0: нет оснований для отклонения гипотезы

Нормальная модель N(μ, σ²), критерий Колмогорова
Δ = 1.002094


p-value = 0.0144
Решение по H0: отвергается

Сравнение результатов:
1) Для равномерного распределения критерий χ² гипотезу не отвергает, а критерий Колмогорова — отвергает.
2) Для нормального распределения критерий χ² также не отвергает гипотезу, а критерий Колмогорова — отвергает.
